# Lagrange-multiplier contact in LowLevelFEM

This short example solves the same two-dimensional Hertz-type contact problem
as the penalty tutorial, but the normal contact constraint is enforced with a
Lagrange multiplier.

The two bodies are initially tangent and the upper body is moved downward in
three equal displacement steps.

For frictionless normal contact,

$$
B =
\int_{\Gamma_c}
N_\lambda^T G\,d\Gamma,
$$

where `G = ContactGap(C)` and `Λ` is a scalar multiplier field.

For the current active contact region the mixed system is

$$
\begin{bmatrix}
K & B^T \\
B & 0
\end{bmatrix}
\begin{bmatrix}
\Delta u \\
\Delta\lambda
\end{bmatrix}
=
-
\begin{bmatrix}
Ku-f+B^T\lambda \\
B(r+u)
\end{bmatrix}.
$$

Unlike the penalty method, no penalty stiffness has to be selected.


In [79]:
using LowLevelFEM, LinearAlgebra, SparseArrays

openGeometry("2D-Hertz.geo")


## Model


In [80]:
matU = Material("upper", E=2e5, ν=0.3)
matL = Material("lower", E=2e5, ν=0.3)

U = Field(
    [matU, matL],
    type=:VectorField,
    dim=2,
    fieldName=:u
)

ε = SymGrad

K =
    ∫(ε(U) ⋅ D(:PlaneStrain, matL) ⋅ ε(U), Ω="lower") +
    ∫(ε(U) ⋅ D(:PlaneStrain, matU) ⋅ ε(U), Ω="upper")

# Zero external load.
f = ∫(U ⋅ [0, 0], Γ="top");


## Contact and multiplier field

The bodies are initially tangent. The master normal points downward, therefore
`normal_sign=-1`.

The scalar field `Λ` represents the normal Lagrange multiplier. With the present
sign convention compression corresponds to a negative multiplier, so the
contact pressure is `-λ`.


In [81]:
support0 = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

u0 = solveField(K, f, support=support0)

C = contact(
    U,
    master="master",
    slave="slave",
    displacement=u0,
    normal_sign=-1,
    activation_tol=1e-8
)

G = ContactGap(C, active=:all)

c_AS = 1e5

# Scalar multiplier field.
Λ = Field(
    [matL],
    type=:ScalarField,
    dim=2,
    fieldName=:λ,
    reducedOrder=true
)

λ = scalarField(Λ, "lower", 0.0)

Tλ, Rλ = reductionMatrices(Λ)

nλr = size(Tλ, 2)

# Reduced multiplier coefficients
λr = zeros(nλr)

# Absolute nodal reference positions.
r = projectTo2D(nodePositionVector(U))

# Zero multiplier block of the KKT system.
Zλ = SystemMatrix(
    spzeros(ndofs(Λ), ndofs(Λ)),
    Λ
)


sparse(Int64[], Int64[], Float64[], 6875, 6875)

## Boundary conditions

Each load step first applies one prescribed downward displacement increment.
The mixed contact correction then uses homogeneous displacement boundary
conditions, so the prescribed displacement remains unchanged.


In [82]:
support_increment = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

function loadIncrementSupport(Δindent)
    [
        BoundaryCondition("top",    ux=0, uy=-Δindent),
        BoundaryCondition("bottom", uy=0),
        BoundaryCondition("left",   ux=0),
        BoundaryCondition("right",  ux=0)
    ]
end

free_u = freeDoFs(U, support_increment)

freeNorm(v) =
    LinearAlgebra.norm(v[free_u])


freeNorm (generic function with 1 method)

## Three displacement steps

The total indentation is divided into three equal increments.

At every contact iteration:

1. `B` is assembled on the currently active Gauss-point contact region,
2. the mixed KKT correction is solved,
3. tensile multipliers are removed because frictionless contact can only
   transmit compression.


In [83]:
indentation_final = 0.40
nsteps = 3
Δindent = indentation_final / nsteps

maxiter = 30
tol = 1e-4
ω = 0.5

# PDAS scaling parameter.
# This is NOT a penalty stiffness.
c_AS = 1e5

# Reduced multiplier space:
# lambda_full = Tλ * lambda_r
Tλ, Rλ = reductionMatrices(Λ)

nλr = size(Tλ, 2)

λr = zeros(nλr)

# Candidate multiplier DOFs belonging to the slave boundary.
Rslave = Rλ[:, C.slave_nodes]

I, _, V = findnz(Rslave)

slave_r = sort!(unique!(
    [
        I[k]
        for k in eachindex(I)
        if !iszero(V[k])
    ]
))
#slave_r = sort!(unique!(
#    I[k] for k in eachindex(I) if !iszero(V[k])
#))

u = copy(u0)

for step in 1:nsteps

    println()
    println("Load step ", step, "/", nsteps)

    # ----------------------------------------------------------
    # Elastic predictor
    # ----------------------------------------------------------

    Δu_load = solveField(
        K,
        f,
        support=loadIncrementSupport(Δindent)
    )

    u_it = u + Δu_load

    active_old = Int[]

    for iter in 1:maxiter

        # ------------------------------------------------------
        # 1. Current nodal gap
        # ------------------------------------------------------

        updateContact!(C, u_it)

        gap = ContactGap(C, u_it)

        # Full nodal gap -> reduced P1 multiplier space.
        gr = Rλ * DoFs(gap)

        # ------------------------------------------------------
        # 2. Primal-dual active set in the reduced space
        #
        # gap >= 0
        # lambda <= 0
        # gap * lambda = 0
        #
        # active if lambda + c_AS * gap < 0
        # ------------------------------------------------------

        active_r = [
            i for i in slave_r
            if λr[i] + c_AS * gr[i] < 0.0
        ]

        isempty(active_r) &&
            error(
                "No active contact was found in load step $step."
            )

        inactive_r =
            setdiff(slave_r, active_r)

        λr[inactive_r] .= 0.0

        # ------------------------------------------------------
        # 3. Full mixed coupling
        # ------------------------------------------------------

        B = ∫(
            Λ ⋅ G;
            threads=:auto
        )

        # ------------------------------------------------------
        # 4. Reduce only the multiplier side
        #
        # B_r = Tλ' * B
        # ------------------------------------------------------

        Br = Tλ' * B.A

        # ------------------------------------------------------
        # 5. Residuals
        # ------------------------------------------------------

        uvec = DoFs(u_it)
        rvec = DoFs(r)
        fvec = DoFs(f)

        ru =
            K.A * uvec -
            fvec +
            Br' * λr

        rλ =
            Br * (rvec + uvec)

        # ------------------------------------------------------
        # 6. Reduced KKT matrix
        # ------------------------------------------------------

        nu = ndofs(U)

        A = [
            K.A                         Br'
            Br    spzeros(nλr, nλr)
        ]

        R = vcat(
            ru,
            rλ
        )

        # Free displacement DOFs +
        # currently active reduced multiplier DOFs.
        free = vcat(
            free_u,
            nu .+ active_r
        )

        # ------------------------------------------------------
        # 7. Solve active reduced KKT system
        # ------------------------------------------------------

        Δx =
            zeros(Float64, nu + nλr)

        Δx[free] =
            -A[free, free] \
             R[free]

        Δu =
            @view Δx[1:nu]

        Δλr =
            @view Δx[nu+1:end]

        # ------------------------------------------------------
        # 8. Error
        # ------------------------------------------------------

        err =
            norm(Δu[free_u]) /
            max(
                norm(uvec[free_u]),
                eps(Float64)
            )

        same_active =
            active_r == active_old

        println(
            "  iter = ", iter,
            ", active P1 multiplier DOFs = ",
            length(active_r),
            ", error = ", err
        )

        # ------------------------------------------------------
        # 9. Relaxed update
        # ------------------------------------------------------

        DoFs(u_it)[:] .+=
            ω .* Δu

        λr .+=
            ω .* Δλr

        if same_active && err < tol
            break
        end

        active_old =
            copy(active_r)
    end

    u = u_it
end


Load step 1/3
  iter = 1, active P1 multiplier DOFs = 37, error = 0.2548862788866843
  iter = 2, active P1 multiplier DOFs = 35, error = 0.14760491808729737
  iter = 3, active P1 multiplier DOFs = 33, error = 0.08991095999324375
  iter = 4, active P1 multiplier DOFs = 29, error = 0.060376279939376755
  iter = 5, active P1 multiplier DOFs = 29, error = 0.04475055775857796
  iter = 6, active P1 multiplier DOFs = 27, error = 0.035023833482011386
  iter = 7, active P1 multiplier DOFs = 25, error = 0.027793795652518317
  iter = 8, active P1 multiplier DOFs = 23, error = 0.021600385227799827
  iter = 9, active P1 multiplier DOFs = 21, error = 0.01585917820926114
  iter = 10, active P1 multiplier DOFs = 19, error = 0.010341350600915462
  iter = 11, active P1 multiplier DOFs = 19, error = 0.005181833736359063
  iter = 12, active P1 multiplier DOFs = 19, error = 0.0025936958688214133
  iter = 13, active P1 multiplier DOFs = 19, error = 0.0012975420900919728
  iter = 14, active P1 multiplier DO

In [84]:
indentation_final = 0.40
nsteps = 3
Δindent = indentation_final / nsteps

maxiter = 30
tol = 1e-4
ω = 0.5

# Active-set parameter.
# This is NOT a penalty stiffness.
c_AS = 1e5

u = copy(u0)

for step in 1:nsteps

    println()
    println("Load step ", step, "/", nsteps)

    # --------------------------------------------------------------
    # Elastic predictor
    # --------------------------------------------------------------

    Δu_load = solveField(
        K,
        f,
        support=loadIncrementSupport(Δindent)
    )

    u_it = u + Δu_load

    active_old = Int[]

    for iter in 1:maxiter

        # ----------------------------------------------------------
        # 1. Update the complete contact geometry
        #
        # We need the nodal gap for the primal-dual active set.
        # ----------------------------------------------------------

        updateContact!(C, u_it)

        gap = ContactGap(C, u_it)

        g =
            DoFs(gap)[C.slave_nodes]

        λs =
            DoFs(λ)[C.slave_nodes]

        # ----------------------------------------------------------
        # 2. Primal-dual active set
        #
        # gap >= 0
        # lambda <= 0
        # gap * lambda = 0
        #
        # A = { i : lambda_i + c_AS * gap_i < 0 }
        # ----------------------------------------------------------

        active_local =
            findall(
                λs .+ c_AS .* g .< 0.0
            )

        active_λ =
            C.slave_nodes[active_local]

        isempty(active_λ) &&
            error(
                "No active contact was found in load step $step."
            )

        inactive_λ =
            setdiff(
                C.slave_nodes,
                active_λ
            )

        # Multipliers are exactly zero outside the active set.
        DoFs(λ)[inactive_λ] .= 0.0

        # ----------------------------------------------------------
        # 3. Mixed contact operator
        #
        # G uses active=:all. The PDAS decides which multiplier
        # equations are retained in the KKT system.
        # ----------------------------------------------------------

        B = ∫(
            Λ ⋅ G;
            threads=:auto
        )

        Br = Tλ' * B.A
        
        # ----------------------------------------------------------
        # 4. Mixed residual
        # ----------------------------------------------------------

        r_u =
            K * u_it -
            f +
            B' * λ

        r_λ =
            B * (r + u_it)

        # ----------------------------------------------------------
        # 5. KKT system
        # ----------------------------------------------------------

        A = SystemMatrix([
            K   B'
            B   Zλ
        ])

        R = SystemVector([
            r_u,
            r_λ
        ])

        λoff = A.offsets[2]

        free = vcat(
            free_u,
            λoff .+ active_λ
        )

        # ----------------------------------------------------------
        # 6. Solve the active KKT system
        # ----------------------------------------------------------

        Δx =
            zeros(Float64, size(A, 1))

        Δx[free] =
            -A.A[free, free] \
             R.a[free, 1]

        Δu =
            @view Δx[1:λoff]

        Δλ =
            @view Δx[λoff+1:end]

        # ----------------------------------------------------------
        # 7. Convergence
        # ----------------------------------------------------------

        err =
            LinearAlgebra.norm(
                Δu[free_u]
            ) /
            max(
                LinearAlgebra.norm(
                    DoFs(u_it)[free_u]
                ),
                eps(Float64)
            )

        same_active =
            active_λ == active_old

        println(
            "  iter = ", iter,
            ", active multiplier nodes = ",
            length(active_λ),
            ", error = ", err
        )

        # ----------------------------------------------------------
        # 8. Relaxed update
        # ----------------------------------------------------------

        DoFs(u_it)[:] .+=
            ω .* Δu

        DoFs(λ)[:] .+=
            ω .* Δλ

        # IMPORTANT:
        # Do NOT clip positive multipliers here.
        # The PDAS will remove them from the active set
        # in the next iteration.

        if same_active && err < tol
            break
        end

        active_old =
            copy(active_λ)
    end

    u = u_it
end


Load step 1/3
  iter = 1, active multiplier nodes = 73, error = 0.25913479872966616
  iter = 2, active multiplier nodes = 67, error = 0.15384273108727864
  iter = 3, active multiplier nodes = 63, error = 0.08898891457804949
  iter = 4, active multiplier nodes = 57, error = 0.05028884501403602
  iter = 5, active multiplier nodes = 59, error = 0.03433393916815616
  iter = 6, active multiplier nodes = 53, error = 0.03490569627036443
  iter = 7, active multiplier nodes = 51, error = 0.02539662641659433
  iter = 8, active multiplier nodes = 49, error = 0.01584125988679659
  iter = 9, active multiplier nodes = 47, error = 0.014095754217802613
  iter = 10, active multiplier nodes = 43, error = 0.015152809202908513
  iter = 11, active multiplier nodes = 41, error = 0.0100466614996734
  iter = 12, active multiplier nodes = 39, error = 0.005683943577922777
  iter = 13, active multiplier nodes = 37, error = 0.003518804823300685
  iter = 14, active multiplier nodes = 37, error = 0.001760640887140

## Stress


In [85]:
ε = [
    ∂x(u[1]),
    ∂y(u[2]),
    ∂y(u[1]) + ∂x(u[2])
]

σ = D(:PlaneStrain, matL) * [ε[1], ε[2], ε[3]]

showElementResults(σ[1], name="σx")
showElementResults(σ[2], name="σy")
showElementResults(σ[3], name="τxy")

showDoFResults(
    √(
        (
            σ[1]^2 +
            σ[2]^2 +
            (σ[1] - σ[2])^2 +
            3σ[3]^2
        ) / 2
    ),
    name="σ von Mises"
)


3

## Gap and contact pressure

After the solution the complete nodal contact state is synchronized once.

The Lagrange multiplier is the contact traction. With the present sign
convention,

$$
p_n = \max(-\lambda,0).
$$

In contrast with the penalty method, the pressure is therefore obtained
directly from the multiplier and does not depend on an artificial contact
stiffness.


In [86]:
λfull = Tλ * λr

DoFs(λ)[:] .= λfull

6875-element view(::Vector{Float64}, :) with eltype Float64:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [87]:
updateContact!(C, u)

gap = ContactGap(C, u; gauss=4)

pressure = mapScalarField(
    x -> max(-x, 0.0),
    λ
)

showDoFResults(
    u,
    name="displacement",
    factor=1,
    visible=false
)

plotOnBeam(
    "slave",
    nodesToElements(gap),
    name="gap"
)

plotOnBeam(
    "slave",
    nodesToElements(pressure),
    name="contact pressure"
)

plotOnBeam(
    "slave",
    nodesToElements(-σ[2]),
    name="contact stress"
)

openPostProcessor()


## Note

This notebook intentionally uses a very simple active-contact iteration to keep
the mixed formulation transparent.

For general loading paths, opening/closing contact and higher-order elements, a
primal-dual active-set, augmented-Lagrangian or mortar formulation is more
robust. Those methods can be added later without changing the basic
`ContactGap` weak-form interface shown here.
